quick analysis of PSPS events from CPUC dashboard


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Clean up accidental quote characters and whitespace in the provided path.
csv_path = Path(str(cpuc_psps_database_file).strip().strip("\"'")).expanduser()
print(f"CSV path: {csv_path}")
print(f"Exists: {csv_path.exists()}")

if not csv_path.exists():
    raise FileNotFoundError(f"Could not find CSV at: {csv_path}")

df_raw = pd.read_csv(csv_path)
print(f"Rows: {len(df_raw):,} | Columns: {len(df_raw.columns)}")
print("Columns:")
print(df_raw.columns.tolist())

df = df_raw.copy()

In [ ]:
cpuc_psps_database_file = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/CA_events_data/cpuc_psps_events_dashboard_downloadedJune2026.csv'




In [ ]:
# Start from a clean copy each run so type conversions are reproducible.
df = df_raw.copy()

# Convert highly numeric object columns to numeric values.
for c in df.columns:
    if pd.api.types.is_object_dtype(df[c]):
        cleaned = df[c].astype(str).str.replace(',', '', regex=False).str.strip()
        as_num = pd.to_numeric(cleaned, errors='coerce')
        if as_num.notna().mean() >= 0.9:
            df[c] = as_num

# Parse only explicit date/time-like columns to datetime.
date_candidates = [c for c in df.columns if re.search(r'date|time', c, re.I)]
for c in date_candidates:
    try:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    except Exception:
        pass

# Create a de-energization duration column in hours when a clear start/end pair exists.
start_col = next((c for c in df.columns if re.search(r'(start|de-?energ.*start)', c, re.I)), None)
end_col = next((c for c in df.columns if re.search(r'(end|restor|re-?energ.*end)', c, re.I)), None)

if start_col and end_col and np.issubdtype(df[start_col].dtype, np.datetime64) and np.issubdtype(df[end_col].dtype, np.datetime64):
    df['deenergization_duration_hours'] = (df[end_col] - df[start_col]).dt.total_seconds() / 3600
    duration_status = f"Computed from '{start_col}' and '{end_col}'"
else:
    duration_status = "Could not infer a clean start/end datetime pair."

print(duration_status)
print()
print("Overall summary (numeric columns):")
display(df.describe(include=[np.number]).T)

print("Overall summary (all columns):")
display(df.describe(include='all').T.head(40))

if 'deenergization_duration_hours' in df.columns:
    dur = df['deenergization_duration_hours'].dropna()
    if len(dur):
        print("De-energization duration statistics (hours):")
        print(dur.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).round(3))
        neg_count = int((dur < 0).sum())
        if neg_count:
            print(f"Warning: {neg_count} events have negative duration (likely data quality/date parsing issues).")
        print()
        print(f"Longest events (top 10):")
        top_cols = [c for c in [start_col, end_col, 'deenergization_duration_hours', 'event_name', 'iou'] if c in df.columns]
        display(df.sort_values('deenergization_duration_hours', ascending=False)[top_cols].head(10))

In [ ]:
df[df['deenergization_duration_hours']<0] #.sort_values('deenergization_duration_hours', ascending=False)

In [ ]:
# Identify likely geography and hazard columns.
area_candidates = [c for c in df.columns if re.search(r'county|city|community|district|zip|area|region|circuit', c, re.I)]
hazard_candidates = [c for c in df.columns if re.search(r'hazard|weather|wind|fire|storm|heat|rain|snow|lightning', c, re.I)]

print("Likely area columns:", area_candidates)
print("Likely hazard columns:", hazard_candidates)
print()

# High-level insights requested.
if 'de_energization_status' in df.columns:
    de_yes = (df['de_energization_status'].astype(str).str.strip().str.lower() == 'yes').sum()
    print(f"Events with actual de-energization: {de_yes}/{len(df)} ({de_yes/len(df):.1%})")

if 'damageshazards' in df.columns:
    hazard_events = (df['damageshazards'].fillna(0) > 0).sum()
    print(f"Events with >0 damages/hazards: {hazard_events}/{len(df)} ({hazard_events/len(df):.1%})")

if 'iou' in df.columns:
    print("Event counts by IOU:")
    display(df['iou'].value_counts(dropna=False).to_frame('event_count'))

if 'de_energization_starting_date' in df.columns and pd.api.types.is_datetime64_any_dtype(df['de_energization_starting_date']):
    yearly = df['de_energization_starting_date'].dt.year.value_counts().sort_index()
    print("De-energization starts by year:")
    display(yearly.to_frame('event_count'))

if 'counties_de_energized' in df.columns:
    print("Top events by number of counties de-energized:")
    cols = [c for c in ['event_name', 'iou', 'counties_de_energized', 'deenergization_duration_hours'] if c in df.columns]
    display(df.sort_values('counties_de_energized', ascending=False)[cols].head(10))
print()

for c in area_candidates[:5]:
    vc = df[c].astype(str).str.strip().replace({'': np.nan}).dropna().value_counts().head(15)
    print(f"Most frequently affected values in '{c}':")
    display(vc.to_frame('event_count'))
    print()

for c in hazard_candidates[:5]:
    vc = df[c].astype(str).str.strip().replace({'': np.nan}).dropna().value_counts().head(20)
    print(f"Hazard distribution in '{c}':")
    display(vc.to_frame('event_count'))
    print()

# Column relationship scan: numeric correlations and duration-by-category patterns.
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    print("Top absolute numeric correlations:")
    corr_pairs = (
        corr.where(~np.eye(corr.shape[0], dtype=bool))
            .stack()
            .rename('corr')
            .abs()
            .sort_values(ascending=False)
            .head(20)
    )
    display(corr_pairs.to_frame())
    print()

if 'deenergization_duration_hours' in df.columns and len(area_candidates):
    c = area_candidates[0]
    tmp = df[[c, 'deenergization_duration_hours']].dropna()
    if len(tmp):
        grouped = tmp.groupby(c)['deenergization_duration_hours'].agg(['count', 'mean', 'median', 'max']).sort_values('count', ascending=False).head(20)
        print(f"Duration statistics by '{c}' (top 20 by event count):")
        display(grouped)
        print()

print("Missingness (% by column):")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
display(missing_pct.to_frame('missing_pct'))

In [ ]:
cwd = Path.cwd()
output_dir = (cwd / 'outputs') if cwd.name == 'notebooks' else (cwd / 'notebooks' / 'outputs')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'cpuc_psps_statistical_analysis.xlsx'


def to_export_frame(obj, index_name=None):
    if isinstance(obj, pd.Series):
        frame = obj.to_frame(name='value')
    else:
        frame = obj.copy()
    if index_name and frame.index.name is None:
        frame.index.name = index_name
    return frame


def add_table(tables, sheet_name, description, data, index_name=None):
    frame = to_export_frame(data, index_name=index_name)
    if frame.empty:
        return
    tables.append(
        {
            'sheet_name': sheet_name,
            'description': description,
            'data': frame,
        }
    )


def make_sheet_name(name, used_names):
    cleaned = re.sub(r'[\\/*?:\[\]]', '_', name).strip() or 'Sheet'
    cleaned = cleaned[:31]
    candidate = cleaned
    counter = 1
    while candidate in used_names:
        suffix = f'_{counter}'
        candidate = f"{cleaned[:31 - len(suffix)]}{suffix}"
        counter += 1
    used_names.add(candidate)
    return candidate


tables_to_export = []

numeric_summary = df.describe(include=[np.number]).T
add_table(
    tables_to_export,
    'numeric_summary',
    'Summary statistics for all numeric CPUC PSPS columns.',
    numeric_summary,
    index_name='column',
)

all_summary = df.describe(include='all').T.head(40)
add_table(
    tables_to_export,
    'all_columns_summary',
    'First 40 rows of the full descriptive summary across all column types.',
    all_summary,
    index_name='column',
)

if 'deenergization_duration_hours' in df.columns:
    dur = df['deenergization_duration_hours'].dropna()
    if len(dur):
        duration_stats = dur.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).round(3)
        add_table(
            tables_to_export,
            'duration_stats',
            'Distribution summary for de-energization duration in hours.',
            duration_stats,
            index_name='statistic',
        )

        top_cols = [c for c in [start_col, end_col, 'deenergization_duration_hours', 'event_name', 'iou'] if c in df.columns]
        longest_events = df.sort_values('deenergization_duration_hours', ascending=False)[top_cols].head(10)
        add_table(
            tables_to_export,
            'longest_events',
            'Ten longest PSPS events ranked by computed de-energization duration.',
            longest_events,
        )

if 'de_energization_status' in df.columns:
    de_mask = df['de_energization_status'].astype(str).str.strip().str.lower() == 'yes'
    de_status_summary = pd.DataFrame(
        {
            'metric': ['actual_deenergization_events', 'total_events', 'share_actual_deenergization'],
            'value': [int(de_mask.sum()), int(len(df)), float(de_mask.mean())],
        }
    )
    add_table(
        tables_to_export,
        'deenergization_status',
        'Counts and share of events with an actual de-energization status of yes.',
        de_status_summary,
    )

if 'damageshazards' in df.columns:
    hazard_mask = df['damageshazards'].fillna(0) > 0
    hazard_summary = pd.DataFrame(
        {
            'metric': ['events_with_positive_damageshazards', 'total_events', 'share_with_positive_damageshazards'],
            'value': [int(hazard_mask.sum()), int(len(df)), float(hazard_mask.mean())],
        }
    )
    add_table(
        tables_to_export,
        'hazard_event_summary',
        'Counts and share of events with damages or hazards greater than zero.',
        hazard_summary,
    )

if 'iou' in df.columns:
    iou_counts = df['iou'].value_counts(dropna=False).to_frame('event_count')
    add_table(
        tables_to_export,
        'iou_event_counts',
        'Event counts grouped by investor-owned utility.',
        iou_counts,
        index_name='iou',
    )

if 'de_energization_starting_date' in df.columns and pd.api.types.is_datetime64_any_dtype(df['de_energization_starting_date']):
    yearly = df['de_energization_starting_date'].dt.year.value_counts().sort_index().to_frame('event_count')
    add_table(
        tables_to_export,
        'events_by_year',
        'Event counts by de-energization starting year.',
        yearly,
        index_name='year',
    )

if 'counties_de_energized' in df.columns:
    cols = [c for c in ['event_name', 'iou', 'counties_de_energized', 'deenergization_duration_hours'] if c in df.columns]
    top_county_events = df.sort_values('counties_de_energized', ascending=False)[cols].head(10)
    add_table(
        tables_to_export,
        'top_county_events',
        'Ten events with the highest number of counties de-energized.',
        top_county_events,
    )

for idx, c in enumerate(area_candidates[:5], start=1):
    vc = df[c].astype(str).str.strip().replace({'': np.nan}).dropna().value_counts().head(15)
    add_table(
        tables_to_export,
        f'area_distribution_{idx}',
        f"Top 15 most frequent values observed in the '{c}' area-related column.",
        vc.to_frame('event_count'),
        index_name=c,
    )

for idx, c in enumerate(hazard_candidates[:5], start=1):
    vc = df[c].astype(str).str.strip().replace({'': np.nan}).dropna().value_counts().head(20)
    add_table(
        tables_to_export,
        f'hazard_distribution_{idx}',
        f"Top 20 value counts observed in the '{c}' hazard-related column.",
        vc.to_frame('event_count'),
        index_name=c,
    )

num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    corr_pairs = (
        corr.where(~np.eye(corr.shape[0], dtype=bool))
        .stack()
        .rename('corr')
        .abs()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
        .rename(columns={'level_0': 'variable_1', 'level_1': 'variable_2'})
    )
    add_table(
        tables_to_export,
        'top_correlations',
        'Top 20 absolute pairwise correlations among numeric columns.',
        corr_pairs,
    )

if 'deenergization_duration_hours' in df.columns and len(area_candidates):
    c = area_candidates[0]
    tmp = df[[c, 'deenergization_duration_hours']].dropna()
    if len(tmp):
        grouped = (
            tmp.groupby(c)['deenergization_duration_hours']
            .agg(['count', 'mean', 'median', 'max'])
            .sort_values('count', ascending=False)
            .head(20)
        )
        add_table(
            tables_to_export,
            'duration_by_area',
            f"Duration statistics grouped by the leading area candidate column '{c}'.",
            grouped,
            index_name=c,
        )

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False).round(2).to_frame('missing_pct')
add_table(
    tables_to_export,
    'missingness',
    'Missing-value percentage for each column in the CPUC PSPS dataset.',
    missing_pct,
    index_name='column',
)

sheet_descriptions = []
used_sheet_names = set()
with pd.ExcelWriter(output_path) as writer:
    for item in tables_to_export:
        sheet_name = make_sheet_name(item['sheet_name'], used_sheet_names)
        item['data'].to_excel(writer, sheet_name=sheet_name, startrow=2)
        worksheet = writer.sheets[sheet_name]
        worksheet.cell(row=1, column=1, value='Description')
        worksheet.cell(row=1, column=2, value=item['description'])
        sheet_descriptions.append({'sheet_name': sheet_name, 'description': item['description']})

print(f'Exported {len(sheet_descriptions)} tables to {output_path}')
display(pd.DataFrame(sheet_descriptions))